# Empirical Tuning Tool (Visual Sweep)

**Purpose**: bound Albumentations intensity visually on a curated subset of frames before training,
so synthetic transforms do not erase tools or invent instrument-like artifacts.

**Input**: 15 to 20 training frames with `challenge_primary == "None"`, stratified by tool-size bin
(`1_tiny`, `2_small`, `4_large`, `0_none`). All frames, masks, and metadata are loaded from GCS
(`gs://laparoscopic-seg-REPLACE_K_NUMBER/trainingFrames/`).

**Output**: per-augmentation visual sweep notes plus a markdown results table for the write-up.


## 1. Install dependencies
Run this cell to install the packages required for this notebook.


In [ ]:
%pip install -q matplotlib tqdm pandas opencv-python-headless albumentations ipywidgets

import albumentations, cv2, matplotlib, pandas as pd, numpy as _np, ipywidgets
print(f"cv2 {cv2.__version__} | matplotlib {matplotlib.__version__} "
      f"| pandas {pd.__version__} | numpy {_np.__version__} "
      f"| albumentations {albumentations.__version__} | ipywidgets {ipywidgets.__version__}")


## 2. Setup and configuration
Reuses `strata_common.py` (same helpers as 01a/01b) to pull frames, masks, the baseline
per-image results CSV, and the annotator sheet from GCS into a local cache. Also fetches
`labelmap.txt` and builds `frame_df` with `challenge_primary` and size-band labels for
section 3. Override the bucket or run id with environment variables if the paths move.


In [ ]:
import os
import sys
from pathlib import Path

# Kernel cwd may be the repo root or 03_tooling/; locate strata_common either way.
_COMMON_DIR = None
for _cand in [Path.cwd(), Path.cwd() / "01_baseline_analysis", Path.cwd().parent / "01_baseline_analysis"]:
	if (_cand / "strata_common.py").exists():
		_COMMON_DIR = _cand
		break
if _COMMON_DIR is None:
	raise ModuleNotFoundError(f"strata_common.py not found from cwd={Path.cwd()}")
sys.path.insert(0, str(_COMMON_DIR.resolve()))

import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import strata_common
importlib.reload(strata_common)
from strata_common import *

# ── GCS sources (frames + baseline metadata used only for size/challenge strata) ──
GCS_BUCKET  = os.getenv("LAP_GCS_BUCKET", "laparoscopic-seg-REPLACE_K_NUMBER")
CKPT_BUCKET = os.getenv("LAP_CKPT_BUCKET", "training-checkpoints-REPLACE_K_NUMBER")
RUN_NAME    = os.getenv("LAP_RUN_NAME", "baseline-seed42-clean_cv_Unet_resnet34_bce_lr1e-04_p20")
RUN_ID      = os.getenv("LAP_RUN_ID", "20260813-232105")
GCS_RUN_DIR = f"gs://{CKPT_BUCKET}/experiments/{RUN_NAME}/{RUN_ID}"
FRAMES_GCS  = f"gs://{GCS_BUCKET}/trainingFrames"
SHEET_ID    = os.getenv("LAP_SHEET_ID", "REPLACE_SHEET_ID")
FOREGROUND_LABEL = os.getenv("LAP_FOREGROUND_LABEL", "instrument")

CACHE_DIR  = Path(os.getenv("LAP_CACHE_DIR", "/tmp/laparoscopic_strata_cache"))
OUTPUT_DIR = Path(os.getenv("LAP_OUTPUT_DIR", "/tmp/laparoscopic_empirical_tuning"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH     = fetch_per_image_results(GCS_RUN_DIR, CACHE_DIR)
ANNOTATIONS_PATH = fetch_annotations(SHEET_ID, CACHE_DIR)
FRAME_DATA_ROOT  = fetch_frame_data(FRAMES_GCS, CACHE_DIR)
IMAGES_DIR       = FRAME_DATA_ROOT / "JPEGImages"
MASKS_DIR        = FRAME_DATA_ROOT / "SegmentationClass"

# labelmap lives next to the frame tree on GCS
LABELMAP_FILE = CACHE_DIR / "labelmap.txt"
if not LABELMAP_FILE.is_file():
	strata_common._gsutil("-q", "cp", f"{FRAMES_GCS}/labelmap.txt", str(LABELMAP_FILE))

def parse_labelmap(path: Path) -> dict:
	"""Return {label_name: (r, g, b)} from Pascal VOC `name:r,g,b::` lines."""
	label_colours = {}
	with open(path) as f:
		for line in f:
			line = line.strip()
			if not line or line.startswith("#"):
				continue
			name, rgb_part, *_ = line.split(":")
			r, g, b = [int(x) for x in rgb_part.split(",")]
			label_colours[name] = (r, g, b)
	return label_colours

label_colours = parse_labelmap(LABELMAP_FILE)
assert FOREGROUND_LABEL in label_colours, (
	f"'{FOREGROUND_LABEL}' not in labelmap. Available: {list(label_colours)}"
)
FG_COLOUR = np.array(label_colours[FOREGROUND_LABEL], dtype=np.uint8)

# paired metadata: challenge_primary + size_cls_label for the curated sample in section 3
frame_df, _, _ = load_and_pair(RESULTS_PATH, ANNOTATIONS_PATH)
frame_df = add_derived_columns(frame_df)
frame_df["challenge_primary"] = frame_df["challenge_tokens"].map(lambda xs: xs[0] if xs else "None")

pool = frame_df[frame_df["challenge_primary"].eq("None")].copy()
print("GCS frames :", FRAMES_GCS)
print("GCS run    :", GCS_RUN_DIR)
print("Images     :", IMAGES_DIR)
print("Masks      :", MASKS_DIR)
print("Labelmap   :", LABELMAP_FILE)
print(f"Foreground : '{FOREGROUND_LABEL}' RGB{tuple(FG_COLOUR)}")
print(f"frame_df   : {len(frame_df):,} rows | challenge_primary==None pool: {len(pool):,}")
print(pool["size_cls_label"].value_counts().sort_index().to_string())
print("Output dir :", OUTPUT_DIR)


## 3. Image selection
Curate 15 to 20 frames with `challenge_primary == "None"` so synthetic degradations are not stacked
on natural challenges. Target composition:

| Stratum | Count | Role |
|---|---:|---|
| `1_tiny` | 8 | Most fragile tools; check whether the augmentation erases the tool |
| `2_small` | 4 | Edge deterioration under the transform |
| `4_large` | 3 | Ensure large tools do not distort unnaturally |
| `0_none` | 3 | Blood/smoke must not create instrument-like hallucinations |

List the selected frame IDs and confirm the stratum counts before sweeping.


In [ ]:
TARGET = {"1_tiny": 8, "2_small": 4, "4_large": 3, "0_none": 3}
SEED = 3

parts = []
for label, n in TARGET.items():
	parts.append(pool[pool["size_cls_label"].eq(label)].sample(n=n, random_state=SEED))
selected = pd.concat(parts).reset_index(drop=True)

print(selected[["key", "size_cls_label", "tool_area_pct"]].to_string(index=False))
print()
print("counts:")
print(selected["size_cls_label"].value_counts().reindex(TARGET.keys()))


## 4. White-out check (visual sweep)
Interactive 1x3 viewer: `Original | Augmented | Augmented + Mask Overlay`.
Push to the failure extreme, step back to the safe zone, then **Save to log**. Section 5 prints the table.


In [ ]:
import cv2
import albumentations as A
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as W

AUG_SEED = 0  # fixed so "dial back 10%" comparisons are stable


def resolve_paths(row):
	"""Map a Colab-style results `image` path onto the local GCS cache.

	Tool frames live under JPEGImages/; 0_none frames live under NoTools/ (no GT mask).
	Missing files are fetched individually from GCS.
	"""
	parts = Path(str(row["image"])).parts
	if "JPEGImages" in parts:
		rel = Path(*parts[parts.index("JPEGImages") + 1 :])
		img = FRAME_DATA_ROOT / "JPEGImages" / rel
		mask = FRAME_DATA_ROOT / "SegmentationClass" / rel.with_suffix(".png")
		if not img.is_file():
			img.parent.mkdir(parents=True, exist_ok=True)
			strata_common._gsutil(
				"-q", "cp", f"{FRAMES_GCS}/JPEGImages/{rel.as_posix()}", str(img),
			)
		if not mask.is_file():
			mask.parent.mkdir(parents=True, exist_ok=True)
			strata_common._gsutil(
				"-q", "cp",
				f"{FRAMES_GCS}/SegmentationClass/{rel.with_suffix('.png').as_posix()}",
				str(mask),
			)
		return img, mask
	if "NoTools" in parts:
		rel = Path(*parts[parts.index("NoTools") + 1 :])
		img = FRAME_DATA_ROOT / "NoTools" / rel
		if not img.is_file():
			img.parent.mkdir(parents=True, exist_ok=True)
			strata_common._gsutil(
				"-q", "cp", f"{FRAMES_GCS}/NoTools/{rel.as_posix()}", str(img),
			)
		return img, None  # no instrument GT for 0_none
	raise ValueError(f"unrecognised image path: {row['image']}")


def load_rgb(path: Path):
	bgr = cv2.imread(str(path))
	if bgr is None:
		raise FileNotFoundError(path)
	return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def load_mask(path, shape_hw):
	"""Binary instrument mask, or zeros when there is no GT (0_none / NoTools)."""
	if path is None:
		return np.zeros(shape_hw, dtype=np.uint8)
	rgb = load_rgb(path)
	return np.all(rgb == FG_COLOUR, axis=-1).astype(np.uint8)


def overlay_mask(image, mask, alpha=0.45):
	out = image.copy()
	red = np.zeros_like(out)
	red[..., 0] = 255
	m = mask.astype(bool)
	out[m] = (alpha * red[m] + (1 - alpha) * out[m]).astype(np.uint8)
	return out


def apply_aug(image, mask, transform):
	"""Apply one transform; spatial augs move the mask, pixel augs leave it fixed."""
	pipe = A.Compose([transform], seed=AUG_SEED)
	out = pipe(image=image, mask=mask)
	return out["image"], out["mask"]


def show_triplet(image, aug_image, aug_mask, title=""):
	# Frames are ~900-1280px wide; a subtle pixel-level change survives at full resolution
	# but gets antialiased away once matplotlib downsamples a small subplot for display.
	# Bigger figure + higher dpi keeps more of the real change visible on screen.
	fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), dpi=110)
	panels = [
		(image, "Original"),
		(aug_image, "Augmented"),
		(overlay_mask(aug_image, aug_mask), "Augmented + Mask"),
	]
	for ax, (im, lab) in zip(axes, panels):
		ax.imshow(im)
		ax.set_title(lab)
		ax.axis("off")
	if title:
		fig.suptitle(title, fontsize=11)
	fig.tight_layout()
	display(fig)
	plt.close(fig)


# One primary knob per aug, pinned to an EXACT value (min==max) rather than a sampled
# range. A range like blur_limit=v (int) or brightness_limit=(0.1, v) makes Albumentations
# draw a random point inside it per call, so two adjacent dial steps can render identically
# (or even regress) by chance; pinning removes that noise so the dial value is what you see.
# Secondary params stay fixed at a mid value while you find the cliff.
# Notes: ideal = safe zone look; max = failure extreme; min = too weak to be useful.
SWEEPS = {
	"MotionBlur": {
		"watch": "1_tiny",
		"param": "blur_limit",
		# Measured on a real 1_tiny frame (RjXqHDUznEY/frame_00470, 1104x720): mean abs diff in
		# a dilated ring around the tool mask only went 1.55 -> 8.70 across the old 3-15 range,
		# vs 1.55 -> 18.98 across 3-55; kernel size is in pixels, and 3-15px is under 1.5% of
		# frame width, near-invisible once the viewer downsamples for display.
		"values": [5, 15, 25, 35, 45, 55],
		"ideal": "Shaft still sharp enough to trace; mild smear only.",
		"min": "Almost no blur vs original - too weak for motion robustness.",
		"max": "FAIL: tiny tool boundary dissolves into tissue; mask outline past the shaft.",
		"build": lambda v: A.MotionBlur(blur_limit=(v, v), p=1.0),
		"label": lambda v: f"blur_limit=({v}, {v})",
	},
	"ColorJitter": {
		"watch": "0_none",
		"param": "saturation",
		"values": [0.1, 0.2, 0.4, 0.5, 0.6, 0.8],
		"ideal": "Tissue looks bloodier but still organic; no fake metal on bare frames.",
		"min": "Barely any hue/sat shift - blood pooling not simulated.",
		"max": "FAIL: dark non-tool patches look metallic (false-positive hallucinations).",
		# albumentations 2.x renamed the ColorJitter args: brightness/contrast/saturation/hue,
		# not brightness_range/contrast_range/saturation_range/hue_range (that was the crash).
		"build": lambda v: A.ColorJitter(
			brightness=(1.0, 1.0),
			contrast=(1.0, 1.0),
			saturation=(1.0 + v, 1.0 + v),
			hue=(-0.1, 0.1),
			p=1.0,
		),
		"label": lambda v: f"saturation=x{1.0 + v:.1f} (hue pinned +/-0.1)",
	},
	"RandomFog": {
		"watch": "1_tiny",
		"param": "fog_coef",
		# Lower end fills in the gap below the old floor of 0.3 so the sweep does not jump
		# straight from clear to already-dense; RandomFog(0.2, 0.9) was worst-case by default.
		"values": [0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 0.9],
		"ideal": "Dense smoke but tiny tool outline still barely visible.",
		"min": "Light haze only - not electrocautery-like.",
		"max": "FAIL: 1_tiny tool fully erased / dissolved into background.",
		"build": lambda v: A.RandomFog(
			fog_coef_range=(float(v), float(v)),
			alpha_coef=0.2,
			p=1.0,
		),
		"label": lambda v: f"fog_coef={v}, alpha_coef=0.2",
	},
	"RandomBrightnessContrast": {
		"watch": "1_tiny",
		"param": "brightness_limit",
		"values": [0.1, 0.2, 0.3, 0.4, 0.5],
		"ideal": "Harsh glare on metal but tip texture/geometry still readable.",
		"min": "Flat lighting - no specular challenge.",
		"max": "FAIL: metallic tip white-out (RGB 255,255,255); geometry gone.",
		"build": lambda v: A.RandomBrightnessContrast(
			brightness_limit=(float(v), float(v)),
			contrast_limit=0.2,
			p=1.0,
		),
		"label": lambda v: f"brightness_limit={v}, contrast_limit=0.2",
	},
	"ElasticTransform": {
		"watch": "4_large",
		"param": "alpha",
		# Widened from 20-60; at sigma=6 that band only spanned diff~9 to diff~24 (measured on
		# a synthetic grid image), too tight to tell steps apart. sigma raised 6 -> 25 too: per
		# the albumentations docs, low sigma gives a high-frequency, noise-like field, higher
		# sigma reads as a coherent, recognizable bend (still well under the 50.0 library default).
		"values": [20.0, 40.0, 60.0, 90.0, 120.0, 160.0],
		"ideal": "Mild non-rigid warp; large tool still a plausible instrument.",
		"min": "Negligible warp - little regularization.",
		"max": "FAIL: large tool edges shear/tear into non-physical shapes.",
		"build": lambda v: A.ElasticTransform(alpha=float(v), sigma=25.0, p=1.0),
		"label": lambda v: f"alpha={v}, sigma=25.0",
	},
}

# Filled by the viewer Save button; section 5 renders this as the write-up table.
SWEEP_LOG = {
	name: {
		"Augmentation": f"A.{name}",
		"Parameter Tested (The Extremes)": "",
		"Visual Impact Observed (The Failure)": "",
		"Selected Value (The Safe Zone)": "",
		"extreme_value": None,  # raw dial value; section 5 maps these into AUG_LEVELS
		"safe_value": None,
	}
	for name in SWEEPS
}

# Prefetch the curated sample so the viewer does not stall on first render.
for _, row in selected.iterrows():
	resolve_paths(row)
print(f"ready: {len(selected)} frames cached under {FRAME_DATA_ROOT}")
print("sweeps:", ", ".join(SWEEPS))


In [ ]:

# Close widgets left over from a previous run of this cell. Re-running while iterating
# (which is exactly what happens while chasing a bug) otherwise leaves old dropdowns and
# their observers alive in the kernel, each still wired to plot().
for _w in globals().get("_EMPIRICAL_TUNING_WIDGETS", []):
	try:
		_w.close()
	except Exception:
		pass

# Belt and braces on top of the close() above: a generation token that every re-run of this
# cell bumps. Any callback captured by an EARLIER run of this cell (including ones from before
# this token existed, or ones .close() failed to fully silence) checks its own captured
# generation against the live one inside plot() and becomes a no-op if it does not match, so
# stale widgets can no longer produce duplicate prints/images no matter how many times this
# cell has been re-run in the current kernel.
globals()["_EMPIRICAL_TUNING_GEN"] = globals().get("_EMPIRICAL_TUNING_GEN", 0) + 1
_MY_GEN = globals()["_EMPIRICAL_TUNING_GEN"]

aug_dd = W.Dropdown(options=list(SWEEPS), description="Aug:")
val_dd = W.Dropdown(description="Value:")
stratum_dd = W.Dropdown(
	options=["1_tiny", "2_small", "4_large", "0_none"],
	description="Stratum:",
)
frame_dd = W.Dropdown(description="Frame:")
show_all = W.Checkbox(value=False, description="All in stratum")
note = W.HTML()
guide = W.HTML(
	"<b>Guide:</b> pick an aug, push the value until the failure note matches what you see, "
	"hit <b>Use current -&gt; extreme</b>, type what broke, step one value back, "
	"hit <b>Use current -&gt; safe</b>, then <b>Save to log</b>. Repeat for each aug; "
	"section 5 prints the table. The plot updates automatically whenever Aug/Value/Stratum/"
	"Frame/\"All in stratum\" changes; <b>Plot</b> only forces a manual refresh."
)
extreme_tb = W.Text(description="Extremes:", placeholder="e.g. fog_coef=(0.5, 0.9)", layout=W.Layout(width="90%"))
failure_tb = W.Textarea(
	description="Failure:",
	placeholder="What broke at the extreme?",
	layout=W.Layout(width="90%", height="60px"),
)
safe_tb = W.Text(description="Safe zone:", placeholder="e.g. fog_coef=(0.3, 0.7)", layout=W.Layout(width="90%"))
btn_layout = W.Layout(width="auto", flex="0 0 auto")
use_val_extreme = W.Button(description="Use current -> extreme", layout=btn_layout)
use_val_safe = W.Button(description="Use current -> safe", layout=btn_layout)
save_btn = W.Button(description="Save to log", button_style="success", layout=btn_layout)
plot_btn = W.Button(description="Plot", button_style="primary", layout=btn_layout)
status = W.HTML()
out = W.Output()

_EMPIRICAL_TUNING_WIDGETS = [
	aug_dd, val_dd, stratum_dd, frame_dd, show_all, note, guide,
	extreme_tb, failure_tb, safe_tb, use_val_extreme, use_val_safe, save_btn, plot_btn, status, out,
]

# True while a callback is pushing several widget changes at once (e.g. switching Aug resets
# Value, Stratum and Frame together). Other observers check this and skip plotting so one
# logical action still triggers exactly one plot() instead of one per widget it touches.
_SYNCING = False


def _frames_for(stratum: str):
	return list(selected.loc[selected["size_cls_label"].eq(stratum), "key"])


def _current_label():
	return SWEEPS[aug_dd.value]["label"](val_dd.value)


def _load_log_into_boxes():
	row = SWEEP_LOG[aug_dd.value]
	extreme_tb.value = row["Parameter Tested (The Extremes)"]
	failure_tb.value = row["Visual Impact Observed (The Failure)"]
	safe_tb.value = row["Selected Value (The Safe Zone)"]


def _set_dropdown(widget, options, value):
	"""Replace options safely when the previous selection is not in the new list."""
	options = list(options)
	chosen = value if value in options else options[0]
	# Assign a one-item list first so the stale value cannot fail validation.
	widget.options = [chosen]
	widget.value = chosen
	widget.options = options
	widget.value = chosen


def _fill_controls_for_aug():
	global _SYNCING
	spec = SWEEPS[aug_dd.value]
	mid = spec["values"][len(spec["values"]) // 2]

	_SYNCING = True
	try:
		_set_dropdown(val_dd, spec["values"], mid)
		val_dd.options = [(spec["label"](v), v) for v in spec["values"]]
		val_dd.value = mid
		stratum_dd.value = spec["watch"]
		_set_dropdown(frame_dd, _frames_for(stratum_dd.value), None)
	finally:
		_SYNCING = False

	note.value = (
		f"<b>Ideal (safe zone):</b> {spec['ideal']}<br>"
		f"<b>Too weak (min):</b> {spec['min']}<br>"
		f"<b>Failure extreme (max):</b> {spec['max']}<br>"
		f"<i>Record extremes at the first clear fail, then step one value back for the safe zone.</i>"
	)
	_load_log_into_boxes()


def _on_stratum_change(_):
	global _SYNCING
	if _SYNCING:
		return
	_SYNCING = True
	try:
		_set_dropdown(frame_dd, _frames_for(stratum_dd.value), None)
	finally:
		_SYNCING = False
	plot()


def _on_aug_change(_):
	_fill_controls_for_aug()
	plot()


def _on_control_change(_):
	if _SYNCING:
		return
	plot()


def plot(_=None):
	if globals().get("_EMPIRICAL_TUNING_GEN") != _MY_GEN:
		return  # stale callback from an earlier run of this cell; ignore it
	with out:
		clear_output(wait=True)
		transform = SWEEPS[aug_dd.value]["build"](val_dd.value)
		keys = _frames_for(stratum_dd.value) if show_all.value else [frame_dd.value]
		print(f"{aug_dd.value} | {_current_label()} | stratum={stratum_dd.value}")
		for key in keys:
			row = selected[selected["key"].eq(key)].iloc[0]
			img_path, mask_path = resolve_paths(row)
			image = load_rgb(img_path)
			mask = load_mask(mask_path, image.shape[:2])
			aug_image, aug_mask = apply_aug(image, mask, transform)
			show_triplet(
				image, aug_image, aug_mask,
				title=f"{key}  [{row['size_cls_label']}]",
			)


def _set_extreme(_):
	extreme_tb.value = _current_label()
	SWEEP_LOG[aug_dd.value]["extreme_value"] = val_dd.value


def _set_safe(_):
	safe_tb.value = _current_label()
	SWEEP_LOG[aug_dd.value]["safe_value"] = val_dd.value


def _save(_):
	prev = SWEEP_LOG[aug_dd.value]
	SWEEP_LOG[aug_dd.value] = {
		"Augmentation": f"A.{aug_dd.value}",
		"Parameter Tested (The Extremes)": extreme_tb.value.strip(),
		"Visual Impact Observed (The Failure)": failure_tb.value.strip(),
		"Selected Value (The Safe Zone)": safe_tb.value.strip(),
		# Keep dial values from Use-current so section 5 can emit AUG_LEVELS numerics.
		"extreme_value": prev.get("extreme_value"),
		"safe_value": prev.get("safe_value"),
	}
	n_done = sum(
		1 for r in SWEEP_LOG.values()
		if r["Parameter Tested (The Extremes)"] and r["Selected Value (The Safe Zone)"]
	)
	status.value = (
		f"<b>Saved {aug_dd.value}</b> ({n_done}/{len(SWEEP_LOG)} complete). "
		"Re-run section 5 to print the table."
	)


aug_dd.observe(_on_aug_change, names="value")
stratum_dd.observe(_on_stratum_change, names="value")
val_dd.observe(_on_control_change, names="value")
frame_dd.observe(_on_control_change, names="value")
show_all.observe(_on_control_change, names="value")
plot_btn.on_click(plot)
use_val_extreme.on_click(_set_extreme)
use_val_safe.on_click(_set_safe)
save_btn.on_click(_save)

display(W.VBox([
	W.HBox([aug_dd, val_dd, plot_btn]),
	W.HBox([stratum_dd, frame_dd, show_all]),
	guide,
	note,
	extreme_tb,
	failure_tb,
	safe_tb,
	W.HBox([use_val_extreme, use_val_safe, save_btn]),
	status,
	out,
]))

_fill_controls_for_aug()
plot()


## 5. Documentation
Prints the write-up table from `SWEEP_LOG` (filled via **Save to log** in section 4).
Incomplete rows show blank cells so you can see what is still missing.


In [ ]:
COLS = [
	"Augmentation",
	"Parameter Tested (The Extremes)",
	"Visual Impact Observed (The Failure)",
	"Selected Value (The Safe Zone)",
]
# Stable order matching the sweep checklist
ORDER = ["MotionBlur", "ColorJitter", "RandomFog", "RandomBrightnessContrast", "ElasticTransform"]
sweep_table = pd.DataFrame([SWEEP_LOG[name] for name in ORDER])[COLS]

display(sweep_table)

out_path = OUTPUT_DIR / "visual_sweep_table.csv"
sweep_table.to_csv(out_path, index=False)
print(f"saved: {out_path}")

# Markdown copy-paste block for the dissertation write-up
print("\n--- markdown ---\n")
header = "| " + " | ".join(COLS) + " |"
sep = "|" + "|".join(["---"] * len(COLS)) + "|"
print(header)
print(sep)
for _, row in sweep_table.iterrows():
	cells = [str(row[c]).replace("|", "\\|") if row[c] else "*(fill)*" for c in COLS]
	print("| " + " | ".join(cells) + " |")

# Paste into 00_model_training AUG_LEVELS (numeric form read by build_train_transform)
import re

def _dial_from_label(name, label):
	"""Recover the dial value from a saved label string if Use-current was skipped."""
	if not label:
		return None
	if name == "MotionBlur":
		m = re.search(r"blur_limit=\((\d+),\s*\d+\)", label)
		return int(m.group(1)) if m else None
	if name == "ColorJitter":
		m = re.search(r"saturation=x([0-9.]+)", label)
		return round(float(m.group(1)) - 1.0, 1) if m else None
	if name == "RandomFog":
		m = re.search(r"fog_coef=([0-9.]+)", label)
		return float(m.group(1)) if m else None
	if name == "RandomBrightnessContrast":
		m = re.search(r"brightness_limit=([0-9.]+)", label)
		return float(m.group(1)) if m else None
	if name == "ElasticTransform":
		m = re.search(r"alpha=([0-9.]+)", label)
		return float(m.group(1)) if m else None
	return None


def _levels_from_dial(name, dial):
	"""Map one aug's dial value onto the AUG_LEVELS keys used by 00_model_training."""
	if dial is None:
		return {}
	if name == "MotionBlur":
		return {"motion_blur": int(dial)}
	if name == "ColorJitter":
		return {"saturation": round(1.0 + float(dial), 1)}
	if name == "RandomFog":
		return {"fog_coef": float(dial), "alpha_coef": 0.2}
	if name == "RandomBrightnessContrast":
		return {"brightness_limit": float(dial), "contrast_limit": 0.2}
	if name == "ElasticTransform":
		return {"elastic_alpha": float(dial), "elastic_sigma": 25.0}
	return {}


def _fmt_num(v):
	if isinstance(v, int) and not isinstance(v, bool):
		return str(v)
	# Match 00_model_training style: 1.2, 0.5, 160.0
	return f"{float(v):.1f}"


def _resolve_dial(name, which):
	row = SWEEP_LOG[name]
	dial = row.get(f"{which}_value")
	if dial is not None:
		return dial
	key = (
		"Parameter Tested (The Extremes)" if which == "extreme"
		else "Selected Value (The Safe Zone)"
	)
	return _dial_from_label(name, row.get(key, ""))


aug_levels = {"safe": {}, "extreme": {}}
missing = []
for name in ORDER:
	for which in ("safe", "extreme"):
		parts = _levels_from_dial(name, _resolve_dial(name, which))
		if not parts:
			missing.append(f"{name}/{which}")
		aug_levels[which].update(parts)

print("\n--- config for 00_model_training ---\n")
print("AUG_LEVELS = {")
for which in ("safe", "extreme"):
	lvl = aug_levels[which]
	# Stable key order matching 00_model_training
	keys = [
		"motion_blur", "saturation", "fog_coef", "alpha_coef",
		"brightness_limit", "contrast_limit",
		"elastic_alpha", "elastic_sigma",
	]
	items = [f'"{k}": {_fmt_num(lvl[k])}' for k in keys if k in lvl]
	# Pack like the training notebook: three logical groups across lines
	if len(items) >= 8:
		line1 = ", ".join(items[:4])
		line2 = ", ".join(items[4:6])
		line3 = ", ".join(items[6:])
		print(f'\t"{which}": {{')
		print(f"\t\t{line1},")
		print(f"\t\t{line2},")
		print(f"\t\t{line3},")
		print("\t},")
	else:
		inner = ", ".join(items)
		print(f'\t"{which}": {{{inner}}},')
print("}")
if missing:
	print(f"\n(incomplete: {', '.join(missing)} - fill via Use current -> Save in section 4)")
